# Operations Performance Analytics

This notebook explores a **synthetic dataset** simulating operational performance across various departments in a company. We'll perform exploratory data analysis (EDA), visualize key metrics, and build predictive models to estimate project success rates (regression) and classify risk levels (classification).

## Dataset Overview

The dataset contains daily records of operational metrics across multiple departments. Each row represents aggregated metrics for a department on a given date.

**Columns:**

| Column | Description |
|-------|-------------|
| `date` | Date of observation (daily). |
| `department` | Department name (Sales, Marketing, IT, HR, Finance, Operations, R&D). |
| `project_count` | Number of active projects. |
| `avg_project_duration` | Average duration of projects (days). |
| `budget_k` | Budget allocated (in thousand dollars). |
| `employee_count` | Number of employees in the department. |
| `actual_spend_k` | Actual spend (in thousand dollars). |
| `revenue_k` | Revenue generated (in thousand dollars). |
| `customer_satisfaction` | Customer satisfaction score (0–100). |
| `project_success_rate` | Percentage of projects delivered on time and within budget (0–100). |
| `risk_level` | Categorical risk level (`low`, `medium`, `high`). |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Load dataset
data = pd.read_csv('../data/operations_performance_synthetic.csv', parse_dates=['date'])

# Display shape and first few rows
(data.shape, data.head())

In [ ]:
# Summary statistics for numeric columns
data.describe(include='all').transpose()

In [ ]:
# Distribution of project count
plt.figure(figsize=(6,4))
sns.histplot(data['project_count'], bins=15, kde=True)
plt.title('Distribution of Project Count')
plt.xlabel('Number of Projects')
plt.ylabel('Frequency')
plt.show()

# Budget vs Actual Spend scatter
plt.figure(figsize=(6,4))
sns.scatterplot(x='budget_k', y='actual_spend_k', hue='department', data=data, alpha=0.7)
plt.title('Budget vs Actual Spend by Department')
plt.xlabel('Budget (k USD)')
plt.ylabel('Actual Spend (k USD)')
plt.legend(title='Department', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Risk level counts
plt.figure(figsize=(4,4))
sns.countplot(x='risk_level', data=data, palette='Set2')
plt.title('Risk Level Distribution')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.show()

In [ ]:
# Compute correlation matrix for numeric features
num_cols = ['project_count','avg_project_duration','budget_k','employee_count','actual_spend_k','revenue_k','customer_satisfaction','project_success_rate']
cor_matrix = data[num_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(cor_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

## Predictive Modeling

In this section, we'll build two predictive models:

1. **Regression**: Estimate the `project_success_rate` using numeric and categorical features.
2. **Classification**: Predict `risk_level` categories.

We'll use train-test splits and evaluate model performance with appropriate metrics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report

# Prepare data for regression
X = data.drop(['project_success_rate','risk_level'], axis=1)
y = data['project_success_rate']

# Identify categorical and numeric columns
cat_features = ['department']
num_features = [col for col in X.columns if col not in ['department','date']]

# Preprocess pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ], remainder='drop'
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
reg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

# Train
reg_model.fit(X_train, y_train)

# Predict
y_pred = reg_model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('Regression Model Performance:
MSE: {:.2f}, R^2: {:.3f}'.format(mse, r2))

In [ ]:
# Prepare data for classification
X_cls = data.drop(['risk_level','project_success_rate'], axis=1)
y_cls = data['risk_level']

# Encode target labels
label_encoder = LabelEncoder()
y_cls_encoded = label_encoder.fit_transform(y_cls)

# Use same preprocessor
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls_encoded, test_size=0.2, random_state=42)

cls_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

# Train
cls_model.fit(X_train_cls, y_train_cls)

# Predict
y_pred_cls = cls_model.predict(X_test_cls)

# Evaluate
acc = accuracy_score(y_test_cls, y_pred_cls)
print('Classification Model Accuracy: {:.3f}'.format(acc))
print('
Classification Report:
')
print(classification_report(y_test_cls, y_pred_cls, target_names=label_encoder.classes_))

## Conclusion

This analysis demonstrates how operational metrics can be used to gain insights into departmental performance. Through exploratory data visualization, we observed patterns in budget allocation, spending behavior, and risk distribution across departments.

We built two predictive models:

- A **Random Forest Regression** model predicting `project_success_rate`, which achieved a reasonable $R^2$ score given the synthetic nature of the data.
- A **Random Forest Classification** model to classify `risk_level`, which attained good accuracy.

These models highlight how data-driven approaches can inform strategic decisions, optimize resource allocation, and mitigate operational risks. The techniques used here can be extended to real-world datasets for deeper business insights.